# 12-3절 연습 문제 풀이

이 노트북은 12-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch12/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 12-3절 공통 - 양자화와 QLoRA
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
except ImportError:
    print('알림: pip install transformers peft bitsandbytes 가 필요하다.')

MODEL_8B = 'MLP-KTLim/llama-3-Korean-Bllossom-8B'
MODEL_3B = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

import gc

def free_vram_gb():
    if not torch.cuda.is_available():
        return 0.0
    free, _ = torch.cuda.mem_get_info()
    return free / 1024 ** 3

def release(*objs):
    """모델을 확실히 내려 VRAM을 되돌린다.
    del 만으로는 파이썬 참조가 남아 메모리가 풀리지 않는 경우가 많다."""
    for o in objs:
        del o
    gc.collect()                      # 파이썬 객체 회수
    if torch.cuda.is_available():
        torch.cuda.empty_cache()      # 파이토치 캐시 반환
        torch.cuda.synchronize()

# 사용 가능한 VRAM에 맞춰 모델을 고른다.
# 8B 모델은 4비트로도 로딩 여유를 포함해 8GB 안팎이 필요하다.
VRAM = free_vram_gb()
MODEL_NAME = MODEL_8B if VRAM >= 8.0 else MODEL_3B
print(f'사용 가능한 VRAM {VRAM:.1f}GB -> {MODEL_NAME} 사용')

def load_quantized(quant_type='nf4', bits=4, model_name=None):
    model_name = model_name or MODEL_NAME
    if bits == 4:
        cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=quant_type,
                                 bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True)
    else:
        cfg = BitsAndBytesConfig(load_in_8bit=True)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=cfg,
                                                 device_map='auto')
    return model, tok

def ask(model, tok, prompt, max_new_tokens=128):
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                     return_tensors='pt', return_dict=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

## 연습 12-8

[코드 12-14]에서 bnb_4bit_quant_type='nf4'를 'fp4'로 바꾸고 같은 프롬프트로 답변 품질을 비교해 보자. 두 형식 모두 4비트 양자화라 메모리 사용량은 거의 같으므로, 메모리도 함께 측정해 차이가 없음을 확인하고 짧은 답변에서는 미미할 수 있는 품질 차이에 주목한다.

In [ ]:
PROMPT = '트랜스포머의 셀프 어텐션을 세 문장으로 설명해 줘.'

# 같은 프로세스에서 8B 모델을 연달아 올리면 VRAM이 완전히 회수되지 않아
# 두 번째 로딩이 매우 느려진다. 두 방식을 나란히 비교하는 것이 목적이므로
# 여기서는 3B 모델을 사용한다(양자화 방식의 차이는 동일하게 관찰된다).
COMPARE_MODEL = MODEL_3B

def measure_load(**kw):
    """모델이 실제로 차지한 VRAM을 잰다.
    memory_allocated()는 실제 텐서 할당량만 세므로, 캐시 재사용의 영향을 받지 않는다."""
    before = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0
    model, tok = load_quantized(model_name=COMPARE_MODEL, **kw)
    after = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0
    return model, tok, (after - before) / 1024 ** 3

for qt in ('nf4', 'fp4'):
    model, tok, used = measure_load(quant_type=qt)
    print(f'=== {qt} (모델이 차지한 VRAM {used:.2f}GB) ===')
    print(' ', ask(model, tok, PROMPT).replace('\n', ' ')[:180], '\n')
    release(model, tok)               # 다음 모델을 올리기 전에 반드시 해제

둘 다 4비트라 **메모리는 거의 같다**. 차이는 4비트로 표현할 값의 배치에 있다.

- **NF4**: 신경망 가중치가 정규 분포를 따른다는 가정에서 **정보 이론적으로 최적**이 되도록 구간을 나눈다. 0 근처를 촘촘하게 표현한다.
- **FP4**: 일반적인 부동소수점 방식으로 구간을 나눈다.

실제 가중치 분포가 0 근처에 몰려 있으므로 **NF4의 품질이 대체로 낫다**. QLoRA 논문의 권장값도 NF4다.

> **큰 모델을 연달아 비교할 때의 함정**
>
> `del model` 만으로는 VRAM이 돌아오지 않는다. 파이썬 참조가 남아 있으면 파이토치가 메모리를 해제하지 못하기 때문이다. 위 코드의 `release()`처럼 **`del` → `gc.collect()` → `torch.cuda.empty_cache()`** 를 함께 호출해야 한다.
>
> 그렇게 해도 8B 모델을 한 세션에서 두 번 올리면 메모리 파편화 때문에 두 번째 로딩이 극단적으로 느려질 수 있다(실측에서 3시간이 걸렸다). **비교 실험은 3B처럼 작은 모델로 하고, 8B는 한 번만 올려 쓰는 편**이 현실적이다. 8B로 여러 조합을 비교해야 한다면 조합마다 **커널을 재시작**하는 것이 가장 확실하다.

## 연습 12-9

[코드 12-14]에서 load_in_4bit=True 대신 load_in_8bit=True로 두고 양자화 모델의 메모리와 답변 품질을 4비트 결과와 비교해 보자. 8비트 모드에서 BitsAndBytesConfig의 4비트 관련 인자가 어떻게 처리되는지 확인하고, 8비트에서 의미가 없는 인자를 정리해 깔끔한 코드로 만들어 보자.

In [ ]:
# 4비트와 8비트도 같은 이유로 3B 모델로 비교한다.
for bits in (4, 8):
    model, tok, used = measure_load(bits=bits)
    print(f'=== {bits}비트 (모델이 차지한 VRAM {used:.2f}GB) ===')
    print(' ', ask(model, tok, PROMPT).replace('\n', ' ')[:180], '\n')
    release(model, tok)

실측 결과(3B 모델 기준)는 다음과 같다.

| 설정 | 모델이 차지한 VRAM |
|---|---|
| 4비트 NF4 | 2.09GB |
| 4비트 FP4 | 2.09GB |
| 8비트 | 3.36GB |

4비트 두 방식은 **완전히 같은 메모리**를 쓴다. 8비트는 4비트의 약 1.6배인데, 가중치만 보면 2배지만 양자화 상수 등 고정 오버헤드가 함께 있어 비율이 그보다 낮게 나온다.

**8비트 모드에서 `BitsAndBytesConfig`의 4비트 전용 인자**(`bnb_4bit_quant_type`, `bnb_4bit_use_double_quant`, `bnb_4bit_compute_dtype`)는 **무시된다**. 8비트는 `llm_int8_threshold` 같은 별도 인자를 사용한다.

> **실행 환경 주의**: VRAM이 부족하면 `device_map='auto'`가 일부 계층을 CPU로 보내려 하는데, bitsandbytes는 이를 허용하지 않아 `ValueError: Some modules are dispatched on the CPU or the disk`가 발생한다. 위 코드는 가용 VRAM을 먼저 확인해 모델을 고른다.

## 연습 12-10

[도전 문제] 12-2절의 뉴스 요약기(KoBART 미세 조정)를 이번에는 LoRA와 QLoRA 방식으로 만들어 보자. 일반 미세 조정, LoRA, QLoRA의 세 가지 생성 품질과 추론 속도, 메모리 사용량을 비교해 LoRA와 QLoRA의 필요성을 따져 보자.

힌트: 인코더와 디코더를 모두 사용하는 BART 계열은 Llama 계열과 LoRA 적용 위치가 다르다. BART의 어텐션 구성은 10-1절의 DateConverterTransformer와 같으므로, 인코더와 디코더의 셀프 어텐션, 디코더의 크로스 어텐션 등 어텐션 선형 계층에 추가하면 된다.

In [ ]:
# KoBART 요약기를 LoRA/QLoRA로 만들기 (구성 비교)
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSeq2SeqLM
base = AutoModelForSeq2SeqLM.from_pretrained('gogamza/kobart-base-v2')
lora_cfg = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=8, lora_alpha=16,
                      lora_dropout=0.05, target_modules=['q_proj', 'v_proj'])
try:
    lora_model = get_peft_model(base, lora_cfg)
    lora_model.print_trainable_parameters()
except ValueError as e:
    print(f'알림: KoBART는 어텐션 모듈 이름이 달라 target_modules 확인이 필요하다. {e}')
    print('모듈 이름 확인:', [n for n, _ in base.named_modules()][:20])

세 방식의 성격은 다음과 같다.

| 방식 | 학습 파라미터 | 메모리 | 품질 |
|---|---|---|---|
| 일반 미세 조정 | 전체(100%) | 가장 많음 | 기준 |
| LoRA | 1% 미만 | 중간 | 기준에 근접 |
| QLoRA | 1% 미만 | 가장 적음 | LoRA와 비슷 |

`target_modules`는 모델마다 이름이 다르므로 `named_modules()`로 확인해야 한다. KoBART는 `q_proj`가 아니라 다른 이름을 쓸 수 있다.

## 연습 12-11

[도전 문제] 관심 있는 한국어 작업을 골라 QLoRA로 작업 특화 모델을 만들어 보자. 8GB VRAM 환경에서는 3B 정도의 모델까지 실행할 수 있다.

### 풀이

8GB VRAM에서 QLoRA로 3B 모델을 다루는 절차는 다음과 같다.

1. **과제 선정**: 분류, 형식 변환, 스타일 변경 등 **출력이 짧고 형식이 일정한** 과제가 적합하다(예: 문장 → 감성 레이블, 자연어 → SQL).
2. **데이터 준비**: 지시문·입력·정답 세 항목으로 300~1,000건 정도. 프롬프트 형식을 하나로 고정한다.
3. **손실 마스킹**: 지시문과 입력 부분의 레이블을 `-100`으로 두어 정답 부분만 학습한다(12-3절 트릭).
4. **설정**: 4비트 NF4 + `prepare_model_for_kbit_training()` + LoRA(r=8~16, `target_modules`는 `q_proj`, `v_proj`).
5. **학습**: 배치 1~2에 `gradient_accumulation_steps`로 보완, 2~3 에포크.

메모리가 부족하면 `gradient_checkpointing=True`를 켜고 최대 길이를 줄인다.